In [122]:
import os, json
from pathlib import Path
from PIL import Image, UnidentifiedImageError
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

In [123]:
INPUT_XLSX   = "pre_data_2.xlsx"
OUTPUT_XLSX  = "pre_data_3.xlsx"
MAP_JSON     = "label_map_category.json"

In [124]:
df = pd.read_excel(INPUT_XLSX)

In [125]:
df.head(5)

,category,id,title,url,path
0,Điện Thoại - Máy Tính Bảng,278237932,Điện thoại Xiaomi Redmi 14C - Hàng chính hãng,https://salt.tikicdn.com/cache/280x280/ts/prod...,images_png/phone/278237932.png
1,Điện Thoại - Máy Tính Bảng,278098703,Điện thoại POCO C75 (8GB/256GB) - Hàng Chính Hãng,https://salt.tikicdn.com/cache/280x280/ts/prod...,images_png/phone/278098703.png
2,Điện Thoại - Máy Tính Bảng,278000720,Điện thoại HONOR X5b Plus 4GB/128GB - Hàng chí...,https://salt.tikicdn.com/cache/280x280/ts/prod...,images_png/phone/278000720.png
3,Điện Thoại - Máy Tính Bảng,277930407,Điện thoại Tecno Spark GO 1 (3GB/64GB) - Hàng ...,https://salt.tikicdn.com/cache/280x280/ts/prod...,images_png/phone/277930407.png
4,Điện Thoại - Máy Tính Bảng,277777809,"Điện thoại Samsung Galaxy A26 5G (8/128GB), Mặ...",https://salt.tikicdn.com/cache/280x280/ts/prod...,images_png/phone/277777809.png


In [126]:
df.isnull().sum().sum()

np.int64(0)

In [127]:
MAX_WORKERS  = 8

In [128]:
if "path" not in df.columns:
    raise ValueError(f"{INPUT_XLSX} không có cột 'path'")

paths = df["path"].astype(str).str.strip().tolist()

def probe_image(p):
    try:
        if not os.path.exists(p):
            return (p, False, False, None, None, None, "missing")
        with Image.open(p) as im:
            im.verify()
        # reopen để đọc size/mode
        with Image.open(p) as im2:
            w, h = im2.size
            mode = im2.mode
            fmt = im2.format
        return (p, True, True, w, h, fmt, "")
    except UnidentifiedImageError:
        return (p, True, False, None, None, None, "unidentified image")
    except Exception as e:
        return (p, True, False, None, None, None, f"error:{type(e).__name__}:{e}")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    for r in ex.map(probe_image, paths):
        results.append(r)

total = len(results)
missing = sum(1 for _, exists, _, _, _, _, _ in results if not exists)
bad = sum(1 for _, exists, openable, _, _, _, _ in results if exists and not openable)

print(f"Tổng ảnh: {total} |Missing: {missing} | Unopenable: {bad}")

if missing or bad:
    print("\nDanh sách ảnh lỗi hoặc thông tin ảnh (tối đa 20 dòng):")
    shown = 0
    for p, exists, openable, w, h, fmt, err in results:
        if (not exists) or (exists and not openable):
            print(f"- {p} | exists={exists} | openable={openable} | err={err}")
            shown += 1
            if shown >= 20:
                print("... (còn nữa)")
                break
else:
    print("Tất cả ảnh tồn tại & mở được bình thường!")

# Thống kê kích thước & định dạng
openable_info = [(fmt, w, h) for _, e, o, w, h, fmt, _ in results if e and o]
if openable_info:
    fmts = {}
    sizes = {}
    for fmt, w, h in openable_info:
        fmts[fmt] = fmts.get(fmt, 0) + 1
        sizes[(w, h)] = sizes.get((w, h), 0) + 1

    print("\nThống kê định dạng ảnh:")
    for fmt, count in fmts.items():
        print(f"- {fmt}: {count} ảnh")

    print("\nKích thước ảnh:")
    for (wh, count) in sorted(sizes.items(), key=lambda x: x[1], reverse=True):
        print(f"- {wh[0]}x{wh[1]}: {count} ảnh")

Tổng ảnh: 500 |Missing: 0 | Unopenable: 0
Tất cả ảnh tồn tại & mở được bình thường!

Thống kê định dạng ảnh:
- PNG: 500 ảnh

Kích thước ảnh:
- 224x224: 500 ảnh


In [129]:
if "category" not in df.columns:
    raise ValueError(f"{INPUT_XLSX} không có cột 'category' để encode.")

# Chuẩn hóa text
df["category"] = df["category"].astype(str).str.strip()
df.loc[df["category"].isin(["", "nan", "None", "null"]), "category"] = "unknown"

# Tạo danh sách lớp (theo thứ tự chữ cái)
classes = sorted(df["category"].unique())
cat2id = {c:i+1 for i, c in enumerate(classes)}
df["category_id"] = df["category"].map(cat2id)

# Lưu mapping
with open(MAP_JSON, "w", encoding="utf-8") as f:
    json.dump({"cat2id": cat2id}, f, ensure_ascii=False, indent=2)

print(f"🗺️ Saved label map: {MAP_JSON} -> {cat2id}")

🗺️ Saved label map: label_map_category.json -> {'Laptop - Máy Vi Tính - Linh kiện': 1, 'Máy Ảnh - Máy Quay Phim': 2, 'Thiết Bị Số - Phụ Kiện Số': 3, 'Điện Thoại - Máy Tính Bảng': 4, 'Điện Tử - Điện Lạnh': 5}


In [130]:
order_front = [c for c in ["category", "category_id", "id", "title", "url", "path"] if c in df.columns]
other_cols = [c for c in df.columns if c not in set(order_front)]
df = df[order_front + other_cols]

In [131]:
df.to_excel(OUTPUT_XLSX, index=False)
print(f"✅ Saved encoded file → {OUTPUT_XLSX} | shape={df.shape}")

✅ Saved encoded file → pre_data_3.xlsx | shape=(500, 6)
